<a href="https://colab.research.google.com/github/nilum2002/Fine-Tune-LLMs-/blob/Main/Gamma_Finetune_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U bitsandbytes==0.49.0
!pip install -q -U peft==0.18.0
!pip install -q -U trl==0.26.2
!pip install -q -U accelerate
!pip install -q -U datasets==4.4.2
!pip install -q -U transformers==4.57.3

In [ ]:
import os
import transformers
import torch
from google.colab import userdata
from datasets import load_dataset
from trl import SFTTrainer
from peft import LoraConfig
from transformers import AutoTokenizer, AutoModelForCausalLM # for generating some text based on decoder based transformer
from transformers import BitsAndBytesConfig, GemmaTokenizer

# Explanation for os.environ["WANDB_DISABLED"] = "false" provided in the chat.

In [ ]:
from google.colab import userdata
import os


os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
model_id = "google/gemma-2-2b"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # all 32 bit weights converts in to 4 bits
    bnb_4bit_quant_type="nf4", # 4-bit normal Float
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id, token = os.environ["HF_TOKEN"])
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config,
                                             token = os.environ["HF_TOKEN"],
                                             device_map={"":0}
)


In [ ]:
# test the model
text = "Quote: Imagination is more,"
device = "cuda:0"
inputs = tokenizer(text, return_tensors="pt").to(device)
# outputs
outputs = model.generate(**inputs, max_new_tokens=200)
print("#"*10)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("#"*10)


The line os.environ["WANDB_DISABLED"] = "false" is used to control the integration with Weights & Biases (W&B), a popular platform for tracking and visualizing machine learning experiments. By setting WANDB_DISABLED to "false", you are explicitly enabling W&B logging. This means that subsequent training or fine-tuning processes will send metrics, hyperparameters, and other relevant data to your W&B project for monitoring, analysis, and comparison of your experiments. If this variable were set to "true", W&B logging would be disabled.





In [ ]:
os.environ["WANDB_DISABLED"] = "false"

In [ ]:
lora_config = LoraConfig(
    r = 8,
    target_modules = ["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    task_type = "CAUSAL_LM"
)

In [ ]:
from datasets import load_dataset

data = load_dataset("Abirate/english_quotes")
data = data.map(lambda samples: tokenizer(samples["quote"]), batched=True)


In [ ]:
data["train"]["quote"]

In [ ]:
def formatting_func(example):
  text = f"Quote:{example["quote"][0]}\nAuthor:{example["author"][0]}"
  return [text]

In [ ]:
trainer = SFTTrainer(
    model = model,
    train_dataset = data["train"],
    args = transformers.TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 2,
        max_steps = 100,
        learning_rate = 2e-4,
        fp16 = False,
        logging_steps = 1,
        output_dir = "outputs",
        optim = "paged_adamw_8bit"
    ),
    peft_config = lora_config,
    formatting_func = formatting_func,
)

In [ ]:
trainer.train()

In [ ]:
# test the model
text = "Quote: A women is like a tea bag;"
device = "cuda:0"
inputs = tokenizer(text, return_tensors="pt").to(device)
# outputs
outputs = model.generate(**inputs, max_new_tokens=500)
print("#"*10)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("#"*10)
